In [7]:
!pip install -qU langchain-huggingface sentence-transformers

In [ ]:
!pip install langchain   tiktoken pypdf  langchain-community

In [9]:
!pip install -qU faiss-cpu langchain-community langchain-core

In [10]:
from langchain_core.documents import Document

# 5 create our list of 5 play documents

# Create our list of 5 player documents with rich metadata
player_docs = [
    Document(
        page_content="Lionel Messi is an Argentine professional footballer considered one of the greatest of all time, famous for his dribbling and playmaking.",
        metadata={"name": "Lionel Messi", "sport": "Football/Soccer", "active": True}
    ),
    Document(
        page_content="LeBron James is an American professional basketball player who has won multiple NBA championships and MVP awards.",
        metadata={"name": "LeBron James", "sport": "Basketball", "active": True}
    ),
    Document(
        page_content="Serena Williams is an American former professional tennis player who won 23 Grand Slam women's singles titles.",
        metadata={"name": "Serena Williams", "sport": "Tennis", "active": False}
    ),
    Document(
        page_content="Babar Azam is a Pakistani international cricketer and a top-order batter, known for his elegant cover drives.",
        metadata={"name": "Babar Azam", "sport": "Cricket", "active": True}
    ),
    Document(
        page_content="Muhammad Ali was an American professional boxer and activist, nicknamed 'The Greatest'.",
        metadata={"name": "Muhammad Ali", "sport": "Boxing", "active": False}
    )
]

print(f"Successfully created {len(player_docs)} documents")

Successfully created 5 documents


In [11]:
# generate embaddings
from langchain_huggingface import HuggingFaceEmbeddings

# 1. Initialize our embedding model (same as before)
embeddings_of_doc = HuggingFaceEmbeddings(
    model_name="BAAI/bge-small-en-v1.5",
    model_kwargs={'device': 'cpu'}, # Change to 'cuda' if using Colab GPU
    encode_kwargs={'normalize_embeddings': True}
)





Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: BAAI/bge-small-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [13]:
# 2 . Create Faiss vector database
from langchain_community.vectorstores import FAISS
vectore_store = FAISS.from_documents(player_docs,embeddings_of_doc)
vectore_store.save_local("Faiss_db")
print("Database is successfully build")

Database is successfully build


In [14]:
# add document

vectore_store.add_documents(player_docs)

['c7ac485c-82fe-474e-9896-a4b0059fb0be',
 '5f18b80c-203a-4b1f-8250-1df80dd496ad',
 '14313642-c11d-4232-ab6d-c32af65110ca',
 'dbb90de2-a6b3-4d0f-8ff0-a72b1946816a',
 '2c78e5b8-1158-45d2-bf78-ec5e7b967069']

In [15]:
#view documents
vectore_store.get(include=['player_docs','embeddings_of_doc','metadata'])

AttributeError: 'FAISS' object has no attribute 'get'

In [16]:
from langchain_community.vectorstores import FAISS

# 1. Load the saved database from your folder
# You must pass the same embedding model you used to create it
# allow_dangerous_deserialization=True is required by LangChain because FAISS uses .pkl files.
# It is completely safe as long as you created the database yourself!
loaded_vector_store = FAISS.load_local(
    folder_path="Faiss_db",
    embeddings=embeddings_of_doc,
    allow_dangerous_deserialization=True
)

# 2. Access the documents stored inside the database's internal dictionary
stored_docs = loaded_vector_store.docstore._dict.values()

print(f"Successfully loaded {len(stored_docs)} documents from the database!\n")

# 3. Loop through and print each document's content and metadata
for i, doc in enumerate(stored_docs, 1):
    print(f"--- Document {i} ---")
    print(f"Content: {doc.page_content}")
    print(f"Metadata: {doc.metadata}\n")

Successfully loaded 5 documents from the database!

--- Document 1 ---
Content: Lionel Messi is an Argentine professional footballer considered one of the greatest of all time, famous for his dribbling and playmaking.
Metadata: {'name': 'Lionel Messi', 'sport': 'Football/Soccer', 'active': True}

--- Document 2 ---
Content: LeBron James is an American professional basketball player who has won multiple NBA championships and MVP awards.
Metadata: {'name': 'LeBron James', 'sport': 'Basketball', 'active': True}

--- Document 3 ---
Content: Serena Williams is an American former professional tennis player who won 23 Grand Slam women's singles titles.
Metadata: {'name': 'Serena Williams', 'sport': 'Tennis', 'active': False}

--- Document 4 ---
Content: Babar Azam is a Pakistani international cricketer and a top-order batter, known for his elegant cover drives.
Metadata: {'name': 'Babar Azam', 'sport': 'Cricket', 'active': True}

--- Document 5 ---
Content: Muhammad Ali was an American profes

In [17]:
# 1. Access the underlying raw FAISS index object
faiss_index = loaded_vector_store.index

# 2. Check how many embeddings (vectors) are stored
total_vectors = faiss_index.ntotal
print(f"Total embeddings stored in the index: {total_vectors}\n")

# 3. Loop through and view the actual numerical arrays
for i in range(total_vectors):
    # reconstruct(i) pulls the raw vector array out of the FAISS memory
    vector = faiss_index.reconstruct(i)

    # Let's also grab the associated document so we know WHO this math belongs to
    doc_id = loaded_vector_store.index_to_docstore_id[i]
    document = loaded_vector_store.docstore.search(doc_id)

    print(f"--- Embedding {i+1} ---")
    print(f"Belongs to Document: {document.metadata.get('name', 'Unknown')}")
    print(f"Total Dimensions: {len(vector)}")

    # We only print the first 5 numbers. Printing all 384+ numbers per document
    # would flood your Colab output screen!
    print(f"Vector Data (first 5 numbers): {vector[:5]}\n")

Total embeddings stored in the index: 5

--- Embedding 1 ---
Belongs to Document: Lionel Messi
Total Dimensions: 384
Vector Data (first 5 numbers): [ 0.01487995  0.07624788  0.04286604 -0.04533291  0.0422247 ]

--- Embedding 2 ---
Belongs to Document: LeBron James
Total Dimensions: 384
Vector Data (first 5 numbers): [-0.05190973  0.00701102  0.02326685 -0.08201353  0.00126939]

--- Embedding 3 ---
Belongs to Document: Serena Williams
Total Dimensions: 384
Vector Data (first 5 numbers): [ 0.01500525  0.00088995 -0.0248897   0.0263212   0.02880735]

--- Embedding 4 ---
Belongs to Document: Babar Azam
Total Dimensions: 384
Vector Data (first 5 numbers): [-0.01752559  0.0419196   0.03683585 -0.0165595   0.00398722]

--- Embedding 5 ---
Belongs to Document: Muhammad Ali
Total Dimensions: 384
Vector Data (first 5 numbers): [-0.04445591  0.07632948 -0.01653678 -0.02423164 -0.03690404]



In [18]:
# search docments
vectore_store.similarity_search(
    query = " who is linol Messi?",
    k=1
)

[Document(id='ed79d410-e8b5-489b-aa81-cf6b52c17a99', metadata={'name': 'Lionel Messi', 'sport': 'Football/Soccer', 'active': True}, page_content='Lionel Messi is an Argentine professional footballer considered one of the greatest of all time, famous for his dribbling and playmaking.')]

In [19]:
vectore_store.similarity_search_with_score(
    query = " who is linol Messi?",
    k=1
)

[(Document(id='ed79d410-e8b5-489b-aa81-cf6b52c17a99', metadata={'name': 'Lionel Messi', 'sport': 'Football/Soccer', 'active': True}, page_content='Lionel Messi is an Argentine professional footballer considered one of the greatest of all time, famous for his dribbling and playmaking.'),
  np.float32(0.37388825))]

In [20]:
vectore_store.similarity_search_with_score(
    query = "",
    filer={'sport': 'Football/Soccer'}
)

[(Document(id='ede8b5e5-0197-471f-9020-2c2c408d45f4', metadata={'name': 'LeBron James', 'sport': 'Basketball', 'active': True}, page_content='LeBron James is an American professional basketball player who has won multiple NBA championships and MVP awards.'),
  np.float32(0.96123344)),
 (Document(id='5f18b80c-203a-4b1f-8250-1df80dd496ad', metadata={'name': 'LeBron James', 'sport': 'Basketball', 'active': True}, page_content='LeBron James is an American professional basketball player who has won multiple NBA championships and MVP awards.'),
  np.float32(0.96123344)),
 (Document(id='ed79d410-e8b5-489b-aa81-cf6b52c17a99', metadata={'name': 'Lionel Messi', 'sport': 'Football/Soccer', 'active': True}, page_content='Lionel Messi is an Argentine professional footballer considered one of the greatest of all time, famous for his dribbling and playmaking.'),
  np.float32(1.0724475)),
 (Document(id='c7ac485c-82fe-474e-9896-a4b0059fb0be', metadata={'name': 'Lionel Messi', 'sport': 'Football/Soccer'